# 6. Interactive diagrams (ipyelk)

`sysml2.diagrams` renders models as **interactive ELK diagrams** in
JupyterLab: pan/zoom, collapsible hierarchy, and click-selection that maps
back to model elements. Layout runs in the browser (elkjs via
[ipyelk](https://github.com/jupyrdf/ipyelk), vendored in `vendor/ipyelk`
with local fixes), so the cells below execute headlessly and the diagrams
lay themselves out when a frontend is attached.

Three views, one dispatcher:

| Function | Shows |
|---|---|
| `structure_diagram` | packages, defs (attribute compartments), nested usages, specialization/typing/connection edges |
| `state_diagram` | hierarchical states, entry markers, labeled transitions |
| `action_diagram` | the succession control-flow graph the interpreter executes |
| `diagram` | picks a view from the element's kind |

In [ ]:
import sysml2
from sysml2 import diagrams

## Structure: definitions, compartments, and relationships

In [ ]:
model = sysml2.loads('''
package Rover {
    part def Wheel { attribute diameter : Real = 0.3; }
    part def Motor { attribute torque : Real = 4.2; }
    abstract part def Platform { attribute mass : Real; }

    part def Rover :> Platform {
        attribute mass : Real :>> Platform::mass = 18.0;
        part wheels : Wheel[6];
        part drive : Motor;
    }

    part mission {
        part rover : Rover;
        part lander;
        connect rover to lander;
    }
}
''')
diagrams.structure_diagram(model)

Blue boxes are definitions («part def») with their attributes as
compartment rows; green boxes are usages. Solid blue arrows are
specializations, dashed green arrows are typings, and plain edges are
connections. **Click a node** and the selection is a qualified name:

In [ ]:
structure = diagrams.structure_diagram(model)

selected: list = []
diagrams.on_select(structure, model, selected.extend)

# simulate a browser click (this is what selecting in the UI does):
structure.view.selection.ids = ["Rover::Rover"]
[f"{type(e).__name__}({e.kind}) {e.qualified_name}" for e in selected]

## State machines

In [ ]:
machine_model = sysml2.loads('''
package Machines {
    state def Player {
        entry; then stopped;
        state stopped;
        transition first stopped accept play then playing;
        state playing {
            entry; then normal;
            state normal;
            transition first normal accept fast then fastForward;
            state fastForward;
            transition first fastForward accept fast then normal;
        }
        transition first playing accept stop then stopped;
        transition first playing accept after 3600.0 then stopped;
    }
}
''')
diagrams.state_diagram(machine_model.find("Machines::Player"))

The black square is the entry marker; nested states render inside their
composite; guards show as `[...]` and time triggers as `after 3600.0`.
The same model drives the simulator -- diagram and execution can never
disagree:

In [ ]:
sim = sysml2.Interpreter(machine_model).simulate(
    "Machines::Player", events=["play", "fast", "stop"])
[str(step) for step in sim.trace]

## Action flow: the graph the interpreter actually executes

In [ ]:
flow_model = sysml2.loads('''
package Ops {
    action def Deploy {
        in tested : Boolean;
        out log : String;
        assign log := "";
        action build { assign log := log + "build>"; }
        action inspect { assign log := log + "inspect>"; }
        action ship { assign log := log + "ship"; }
        action abort { assign log := log + "ABORT"; }
        first start then build;
        first build then d1;
        decide d1;
        if tested then inspect;
        else abort;
        first inspect then ship;
        first ship then done;
        first abort then done;
    }
}
''')
deploy = flow_model.find("Ops::Deploy")
diagrams.action_diagram(deploy)

In [ ]:
interp = sysml2.Interpreter(flow_model)
print("tested=true: ", interp.run_action(deploy, inputs={"tested": True}).outputs)
print("tested=false:", interp.run_action(deploy, inputs={"tested": False}).outputs)

## One dispatcher, and a full example

`diagrams.diagram(...)` picks the right view from the element kind. Try it
on the drone example shipped with the repo:

In [ ]:
drone = sysml2.load("../examples/drone.sysml")
diagrams.diagram(drone.find("Drone::FlightStates"))

In [ ]:
diagrams.diagram(drone)   # falls back to the structure view

## Exporting images

The same views render headlessly to SVG/PNG -- the vendored elkjs runs the
layout in a node subprocess instead of the browser (`sysml2.render`):

In [ ]:
from pathlib import Path
import tempfile

from sysml2 import render

out = Path(tempfile.mkdtemp())
render.to_svg(diagrams.state_diagram(machine_model.find("Machines::Player")),
              out / "player.svg")
print((out / "player.svg").stat().st_size, "bytes of SVG")

try:
    render.to_png(drone.find("Drone::FlightStates"), out / "flight.png")
    print("PNG written:", out / "flight.png")
except Exception as err:   # cairosvg needs the native cairo library
    print("PNG skipped:", err)

> **Note** -- ipyelk is vendored (`vendor/ipyelk`, BSD-3-Clause) and
> installed editable so it can be patched as needed; see
> `vendor/ipyelk/README.vendor.md`. Current local fixes: headless-safe
> pipeline scheduling (no `RuntimeError: no running event loop` outside
> Jupyter) and the prebuilt JupyterLab extension grafted from the 2.1.1
> wheel.

## Replaying a simulation

`sysml2.replay` animates an execution trace over the exported state
diagram (needs the `replay` extra: `pip install "sysml2[replay]"`).
`replay_widget` simulates the machine with the same event protocol as
`Interpreter.simulate` (names send events, numbers advance the clock),
bakes the diagram to SVG, and replays the recorded timeline in the
browser: active states light up green (composite ancestors in a
lighter tint), fired transitions pulse orange, and the controls
play/scrub through sim time -- or through the step index when no time
passes at all. The robot below runs a 30-second cleaning mission:
something moves every few seconds, so just press play.

In [ ]:
from sysml2 import replay

robot_model = sysml2.loads('''
package Robots {
    state def CleaningRobot {
        entry; then docked;
        state docked;
        transition first docked accept start then cleaning;
        state cleaning {
            entry; then sweeping;
            state sweeping;
            transition first sweeping accept after 6.0 then mopping;
            state mopping;
            transition first mopping accept after 4.0 then sweeping;
        }
        transition first cleaning accept after 15.0 then charging;
        state charging;
        transition first charging accept after 8.0 then docked;
    }
}
''')

replay.replay_widget(sysml2.Interpreter(robot_model),
                     "Robots::CleaningRobot", events=["start", 30.0])